# Bangkok Transit Map Project

This notebook implements a graph-based representation of the Bangkok transit network and provides pathfinding algorithms.

In [ ]:
import re
import csv
import math
import heapq
from collections import defaultdict, deque

# File path
SQL_FILE_PATH = 'bangkok_railway.sql'

## 1. Data Parsing and Export

We parse the SQL file, merge duplicate interchange stations, and export the data to CSV files using station codes as IDs.

In [ ]:
def parse_and_save_sql(file_path):
    raw_stations = {}
    raw_edges = []
    lines = {}
    raw_fares = []
    
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
        
    # Parse Lines
    line_pattern = re.compile(r"INSERT INTO `line`.*?VALUES\s*(.*?);", re.DOTALL)
    line_matches = line_pattern.findall(content)
    for match in line_matches:
        rows = match.split('),')
        for row in rows:
            row = row.strip().strip('()')
            parts = [p.strip().strip("'") for p in row.split(',')]
            if len(parts) >= 2:
                line_id = int(parts[0])
                line_name = parts[1]
                lines[line_id] = line_name

    # Parse Stations (Raw)
    station_pattern = re.compile(r"INSERT INTO `station`.*?VALUES\s*(.*?);", re.DOTALL)
    station_matches = station_pattern.findall(content)
    for match in station_matches:
        rows = match.split('),')
        for row in rows:
            row = row.strip().strip('()')
            parts = [p.strip().strip("'") for p in row.split(',')]
            if len(parts) >= 9:
                station_id = int(parts[0])
                code = parts[1]
                name = parts[3]
                line_id = int(parts[5])
                try:
                    x = float(parts[7])
                    y = float(parts[8])
                except ValueError:
                    x, y = 0.0, 0.0
                
                raw_stations[station_id] = {
                    'id': station_id,
                    'code': code,
                    'name': name,
                    'line_id': line_id,
                    'line_name': lines.get(line_id, 'Unknown'),
                    'pos': (x, y)
                }

    # Parse Edges (Raw)
    edge_pattern = re.compile(r"INSERT INTO `edge`.*?VALUES\s*(.*?);", re.DOTALL)
    edge_matches = edge_pattern.findall(content)
    for match in edge_matches:
        rows = match.split('),')
        for row in rows:
            row = row.strip().strip('()')
            parts = [p.strip().strip("'") for p in row.split(',')]
            if len(parts) >= 4:
                from_id = int(parts[1])
                to_id = int(parts[2])
                edge_type = parts[3]
                raw_edges.append({'from': from_id, 'to': to_id, 'type': edge_type})

    # Parse Fares
    fare_pattern = re.compile(r"INSERT INTO `station_fare`.*?VALUES\s*(.*?);", re.DOTALL)
    fare_matches = fare_pattern.findall(content)
    for match in fare_matches:
        rows = match.split('),')
        for row in rows:
            row = row.strip().strip('()')
            parts = [p.strip().strip("'") for p in row.split(',')]
            if len(parts) >= 4:
                from_code = parts[1]
                to_code = parts[2]
                try:
                    fare = float(parts[3])
                except ValueError:
                    fare = 0.0
                raw_fares.append({'from_code': from_code, 'to_code': to_code, 'fare': fare})

    # --- Merge Stations Logic ---
    merged_stations = {}
    name_to_id = {}
    sql_id_to_merged_id = {}
    merged_id_to_codes = defaultdict(list)
    
    # First pass: Create merged stations and assign Canonical IDs (using first code found)
    for s_id, s_data in raw_stations.items():
        name = s_data['name']
        code = s_data['code']
        
        if name not in name_to_id:
            # New merged station, use this code as the ID
            if name == 'Siam':
                canonical_id = 'CEN'
            else:
                canonical_id = code
            name_to_id[name] = canonical_id
            merged_stations[canonical_id] = {
                'id': canonical_id,
                'name': name,
                'codes': [code],
                'lines': [s_data['line_name']],
                'pos': s_data['pos']
            }
        else:
            # Existing merged station
            canonical_id = name_to_id[name]
            if code not in merged_stations[canonical_id]['codes']:
                merged_stations[canonical_id]['codes'].append(code)
            if s_data['line_name'] not in merged_stations[canonical_id]['lines']:
                merged_stations[canonical_id]['lines'].append(s_data['line_name'])
        
        sql_id_to_merged_id[s_id] = name_to_id[name]
        merged_id_to_codes[name_to_id[name]].append(code)

    # Pre-calculate Fares between Merged Stations
    code_to_merged_id = {}
    for m_id, codes in merged_id_to_codes.items():
        for c in codes:
            code_to_merged_id[c] = m_id
            
    merged_fare_map = {}
    for f in raw_fares:
        u_code = f['from_code']
        v_code = f['to_code']
        
        if u_code in code_to_merged_id and v_code in code_to_merged_id:
            u_merged = code_to_merged_id[u_code]
            v_merged = code_to_merged_id[v_code]
            
            if u_merged == v_merged:
                continue
                
            fare = f['fare']
            key = (u_merged, v_merged)
            
            if key not in merged_fare_map:
                merged_fare_map[key] = fare
            else:
                merged_fare_map[key] = min(merged_fare_map[key], fare)
                
    final_fares = []
    for (u, v), fare in merged_fare_map.items():
        final_fares.append({'from_id': u, 'to_id': v, 'fare': fare})

    # Second pass: Create merged edges
    final_edges = []
    seen_edges = set()
    
    for edge in raw_edges:
        u_sql = edge['from']
        v_sql = edge['to']
        
        if u_sql not in sql_id_to_merged_id or v_sql not in sql_id_to_merged_id:
            continue
            
        u_merged = sql_id_to_merged_id[u_sql]
        v_merged = sql_id_to_merged_id[v_sql]
        
        if u_merged == v_merged:
            continue
            
        edge_key = (u_merged, v_merged)
        if edge_key in seen_edges:
            continue
        seen_edges.add(edge_key)
        
        fare = merged_fare_map.get((u_merged, v_merged))
        if fare is None:
            if edge['type'] == 'transfer':
                fare = 0.0
            else:
                fare = 15.0 # Default
        
        final_edges.append({
            'from_id': u_merged,
            'to_id': v_merged,
            'type': edge['type'],
            'cost': fare
        })

    # Save to CSV
    with open('stations.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['id', 'name', 'code', 'line_name', 'x', 'y'])
        for s in merged_stations.values():
            writer.writerow([s['id'], s['name'], s['codes'][0], "; ".join(s['lines']), s['pos'][0], s['pos'][1]])

    with open('edges.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['from_id', 'to_id', 'type', 'cost'])
        for e in final_edges:
            writer.writerow([e['from_id'], e['to_id'], e['type'], e['cost']])

    with open('lines.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['id', 'name'])
        for l_id, l_name in lines.items():
            writer.writerow([l_id, l_name])
            
    with open('fares.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['from_id', 'to_id', 'fare'])
        for fare in final_fares:
            writer.writerow([fare['from_id'], fare['to_id'], fare['fare']])
        
    print("Data saved to stations.csv, edges.csv, lines.csv, fares.csv")
    print(f"Merged {len(raw_stations)} raw stations into {len(merged_stations)} unique stations.")

parse_and_save_sql(SQL_FILE_PATH)

## 2. Graph Construction

Load data from CSV and build the graph.

In [ ]:
def load_data_csv():
    stations = {}
    edges = []
    fares = {}
    
    with open('stations.csv', 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            s_id = row['id'] # ID is now string (code)
            stations[s_id] = {
                'id': s_id,
                'name': row['name'],
                'code': row['code'],
                'line': row['line_name'],
                'pos': (float(row['x']), float(row['y']))
            }
            
    with open('edges.csv', 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            edges.append({
                'from': row['from_id'],
                'to': row['to_id'],
                'type': row['type'],
                'cost': float(row['cost'])
            })
            
    with open('fares.csv', 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            fares[(row['from_id'], row['to_id'])] = float(row['fare'])
            
    return stations, edges, fares

class Graph:
    def __init__(self):
        self.nodes = {}
        self.adjacency_list = defaultdict(list)

    def add_node(self, node_id, **kwargs):
        self.nodes[node_id] = kwargs

    def add_edge(self, u, v, weight=1.0, cost=0.0, type='ride'):
        self.adjacency_list[u].append({'to': v, 'weight': weight, 'cost': cost, 'type': type})

    def get_neighbors(self, u):
        return self.adjacency_list[u]

    def get_node(self, u):
        return self.nodes.get(u)

stations_data, edges_data, fares_data = load_data_csv()
transit_graph = Graph()

# Add Nodes
for s_id, s_info in stations_data.items():
    transit_graph.add_node(s_id, name=s_info['name'], code=s_info['code'], line=s_info['line'], pos=s_info['pos'])

# Add Edges
for edge in edges_data:
    from_id = edge['from']
    to_id = edge['to']
    
    if from_id not in transit_graph.nodes or to_id not in transit_graph.nodes:
        continue
        
    cost = edge['cost']
    if edge['type'] == 'transfer':
        cost = 0.0
        
    transit_graph.add_edge(from_id, to_id, weight=1.0, cost=cost, type=edge['type'])

## 3. Algorithms

Implement Pathfinding Algorithms with corrected cost calculation.

In [ ]:
def calculate_path_cost(path, graph, fares_data):
    """Calculates the total fare for a given path based on line segments."""
    total_cost = 0.0
    if not path or len(path) < 2:
        return 0.0
        
    # Start tracking the first segment
    segment_start = path[0]
    
    for i in range(1, len(path)):
        u = path[i-1]
        v = path[i]
        
        # Determine the type of edge (ride or transfer)
        edge_type = 'ride'
        for edge in graph.get_neighbors(u):
            if edge['to'] == v:
                edge_type = edge['type']
                break
        
        # If we encounter a transfer, the current segment ends
        if edge_type == 'transfer':
            # Calculate fare for the segment just finished
            if segment_start != u:
                fare = fares_data.get((segment_start, u), 0.0)
                # Try reverse direction if not found
                if fare == 0.0 and segment_start != u:
                     fare = fares_data.get((u, segment_start), 0.0)
                total_cost += fare
            
            # Transfers themselves are free (cost 0), start new segment from next station
            segment_start = v
        else:
            # Continue the current ride segment
            pass
            
    # Calculate fare for the final segment after the loop
    if segment_start != path[-1]:
        fare = fares_data.get((segment_start, path[-1]), 0.0)
        if fare == 0.0:
             fare = fares_data.get((path[-1], segment_start), 0.0)
        total_cost += fare
        
    return total_cost

def reconstruct_path(came_from, current, graph, fares_data):
    """Reconstructs the path from start to end using the came_from map."""
    path = [current]
    # Backtrack from end to start
    while current in came_from:
        current = came_from[current]
        path.append(current)
    path.reverse() # Reverse to get start -> end
    
    # Create a list of station details for the result
    result = []
    for i in range(len(path)):
        node_id = path[i]
        node_info = graph.get_node(node_id)
        step = {'station': node_info['name'], 'line': node_info['line']}
        result.append(step)
    
    total_stations = len(path)
    # Calculate the actual fare for this reconstructed path
    total_cost = calculate_path_cost(path, graph, fares_data)
    return result, total_stations, total_cost

def dijkstra(graph, start_id, end_id, weight_key='weight'):
    """Finds the cheapest path using Dijkstra's algorithm."""
    # Priority queue stores (current_cost, current_node)
    queue = [(0, start_id)]
    # Track minimum cost to reach each node
    distances = {node: float('inf') for node in graph.nodes}
    distances[start_id] = 0
    came_from = {} # To reconstruct the path
    
    while queue:
        # Pop the node with the lowest cost so far
        current_dist, current_node = heapq.heappop(queue)
        
        # If we reached the destination, we are done
        if current_node == end_id:
            return reconstruct_path(came_from, current_node, graph, fares_data)
        
        # Skip if we found a shorter path to this node already
        if current_dist > distances[current_node]:
            continue
            
        # Explore neighbors
        for edge in graph.get_neighbors(current_node):
            neighbor = edge['to']
            weight = edge[weight_key] # Use 'cost' for cheapest path
            new_dist = current_dist + weight
            
            # If a cheaper path is found, update distance and push to queue
            if new_dist < distances[neighbor]:
                distances[neighbor] = new_dist
                came_from[neighbor] = current_node
                heapq.heappush(queue, (new_dist, neighbor))
                
    return None, 0, 0

def bfs_shortest_path(graph, start_id, end_id):
    """Finds the shortest path (fewest hops) using Breadth-First Search."""
    queue = deque([start_id])
    visited = {start_id}
    came_from = {}
    
    while queue:
        current = queue.popleft()
        
        # If destination reached, reconstruct path
        if current == end_id:
            return reconstruct_path(came_from, current, graph, fares_data)
            
        # Visit all unvisited neighbors
        for edge in graph.get_neighbors(current):
            neighbor = edge['to']
            if neighbor not in visited:
                visited.add(neighbor)
                came_from[neighbor] = current
                queue.append(neighbor)
    return None, 0, 0

def find_any_path(graph, start_id, end_id):
    """Finds any valid path (uses BFS for simplicity)."""
    return bfs_shortest_path(graph, start_id, end_id)

def find_all_paths(graph, start_id, end_id, limit=10):
    """Finds all simple paths using Depth-First Search (DFS)."""
    # limit: max number of paths to return to avoid explosion
    paths = []
    # Stack stores (current_node, current_path_list)
    stack = [(start_id, [start_id])]
    
    while stack and len(paths) < limit:
        (vertex, path) = stack.pop()
        
        # Explore neighbors
        for edge in graph.get_neighbors(vertex):
            neighbor = edge['to']
            
            # If destination found, add to results
            if neighbor == end_id:
                full_path = path + [neighbor]
                
                # Build result details
                result = []
                for i in range(len(full_path)):
                    node_info = graph.get_node(full_path[i])
                    result.append({'station': node_info['name'], 'line': node_info['line']})
                
                total_cost = calculate_path_cost(full_path, graph, fares_data)
                paths.append((result, len(full_path), total_cost))
                
            # If neighbor not in current path (avoid cycles), continue DFS
            elif neighbor not in path:
                stack.append((neighbor, path + [neighbor]))
                
    return paths

## 4. Testing

Run tests to verify the algorithms.

In [ ]:
def get_station_id_by_name(name):
    for s_id, info in stations_data.items():
        if info['name'].lower() == name.lower():
            return s_id # Return string ID
    return None

def run_test(source_name, dest_name):
    start_id = get_station_id_by_name(source_name)
    end_id = get_station_id_by_name(dest_name)
    
    if start_id is None or end_id is None:
        print("Invalid station names")
        return
        
    print(f"--- Path from {source_name} to {dest_name} ---")
    
    # Shortest Path (Least Hops)
    path, stations_count, cost = bfs_shortest_path(transit_graph, start_id, end_id)
    print(f"Shortest Path (Stations: {stations_count}, Cost: {cost:.2f}):")
    if path:
        print(" -> ".join([p['station'] for p in path]))
    else:
        print("No path found")
        
    # Cheapest Path
    path, stations_count, cost = dijkstra(transit_graph, start_id, end_id, weight_key='cost')
    print(f"\nCheapest Path (Stations: {stations_count}, Cost: {cost:.2f}):")
    if path:
        print(" -> ".join([p['station'] for p in path]))
        
    # Any Path
    path, stations_count, cost = find_any_path(transit_graph, start_id, end_id)
    print(f"\nAny Path:")
    if path:
        print(" -> ".join([p['station'] for p in path]))
        
    # All Paths
    print(f"\nAll Paths (Top 5):")
    all_paths = find_all_paths(transit_graph, start_id, end_id, limit=5)
    for i, (path, count, cost) in enumerate(all_paths):
        print(f"Path {i+1} (Stations: {count}, Cost: {cost:.2f}):")
        print(" -> ".join([p['station'] for p in path]))

# Test 1: Same Line (Sukhumvit -> Asok)
run_test('Siam', 'Asok')

# Test 2: Transfer (Siam -> Silom)
run_test('Siam', 'Si Lom')